# Sesión 4 · Transformación de datos — empleos (versión CON cola larga (outliers conservados))

**Curso:** Minería de Datos · UNIMINUTO Ibagué · 2026-2
**Ingreso:** salida de `3.0_preprocesamiento.ipynb` · **Paso (CRISP-DM):** Preparación de datos → *transformación*.

> Regla de oro de la sesión: **no se puede entrenar una IA con datos no numéricos**. Los valores categóricos, si se entregan tal cual, la sesgan. Aquí llevamos cada columna a una representación numérica *honesta* (one-hot u ordinal) y luego **escalamos** las numéricas para que ninguna columna "secuestre" las distancias que usarán KNN / K-means en sesiones posteriores.
>
> Desde la integración de ESCO el dataset trae además **competencias por oficio**: la sección **2.5** las pasa a número, deja por qué de las seis entran solo dos a la `X`, y documenta el costo de hacerlo (aquí se pierden 28.304 filas, el 7,52 % sin oficio clasificado).
>
> Este cuaderno es **transversal a las dos versiones** de la sesión 3. En la celda de configuración de la sección 0 se elige `con_outliers` (376.567 filas) o `sin_outliers` (341.090 filas). La sección 4 re-audita los outliers en tiempo de ejecución para decidir el escalador.

## 0 · Entorno, versión y carga

```text
Limpio -> Codificar categóricas -> Escalar -> (PCA si aplica) -> Modelar
```
Hoy cubrimos los tres primeros bloques.


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.preprocessing import (
    KBinsDiscretizer, MinMaxScaler, RobustScaler, StandardScaler, OneHotEncoder,
)
from sklearn.decomposition import PCA

print("Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| sklearn", __import__("sklearn").__version__)

# Regla D5: la raiz se resuelve buscando 'data/' hacia arriba; nunca una ruta
# absoluta de esta maquina (mismo criterio que src/limpieza/*.py y que 2.0 / 3.0).
RAIZ = Path.cwd()
while not (RAIZ / "data").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
PROC = RAIZ / "data" / "03_processed"

# <<< CONFIG: cambie el valor de VERSION para elegir la versión del dataset >>>
VERSION = "con_outliers"          # "con_outliers"  o  "sin_outliers"
CSV = PROC / ("empleos_limpio.csv" if VERSION == "con_outliers"
              else "empleos_limpio_sin_outliers.csv")
print("\nRaíz del proyecto:", RAIZ)
print("Versión:", VERSION, "| archivo:", CSV.name)

SAMPLE_SIZE = 50_000   # submuestra para las matrices densas de dummies (seccion 3)
RNG = np.random.RandomState(42)
N_PARES = 300          # parejas de filas para medir el aporte a la distancia (4.2)
SEED_DIST = 42         # semilla de esas parejas (reproducibilidad, regla D5)

Python 3.14.7 | pandas 3.0.2 | numpy 2.4.4 | sklearn 1.9.1

Raíz del proyecto: C:\Users\ACER\Desktop\MineriaProtect
Versión: con_outliers | archivo: empleos_limpio.csv


In [2]:
df = pd.read_csv(
    CSV, dtype=str, encoding="utf-8", keep_default_na=False, na_values=[""]
)
print(f"dataset           : {df.shape[0]:,} filas x {df.shape[1]} columnas | "
      f"{df['resume_id'].nunique():,} personas")
print("\nNulos por columna (los NaN de ocupacion ya estan explicados por banderas en la sesion 3):")
print(df.isna().sum()[df.isna().sum() > 0].to_string())

dataset           : 376,567 filas x 20 columnas | 71,061 personas

Nulos por columna (los NaN de ocupacion ya estan explicados por banderas en la sesion 3):
occupation_code        28304
occupation_label       28304
isco_group             25805
isco_group_label       25805
isco_level             25805
n_skills_essential     28304
n_skills_competence    28304
n_skills_knowledge     28304
n_skills_sin_tipo      28304
veces_ese_oficio       28304


---
## 1 · ¿Por qué cada columna es *one-hot* u *ordinal*?

La naturaleza de la variable decide la codificación. La regla del curso:

- **Nominales** (sin jerarquía natural) → **one-hot**: "aseo, libros, licores" no tienen orden, no se puede decir que uno "vale más" que otro. Numerarlos 1, 2, 3 inventaría distancia.
- **Ordinales** (jerarquía real y verificable) → **ordinal / KBins**: la duración de un empleo o el nivel educativo sí tienen orden ("jefe > auxiliar").

Las cifras de filas dependen de la versión elegida; todo se calcula en tiempo de ejecución.

| Columna | Naturaleza | Codificación elegida | Por qué |
|---|---|---|---|
| `start_date` / `end_date` | **Ordinal** (tiempo: Q1 1965 < Q1 2026) | ordinal (año×4+q) → `KBinsDiscretizer` (demo) + escalado | El tiempo tiene orden estricto y significado; jamás one-hot (≈206 categorías y se perdería el orden) |
| `university_level` | **Ordinal** (jerarquía de estudios) | mapeo ordinal manual 0–4 | "No reportado < Secondary school < Bachelor < Master < PhD" es jerarquía real; son solo 5 valores discretos, mapeo directo (KBins no aplica a 5 niveles fijos) |
| `isco_level` | **Ordinal por diseño** (nivel 1–4 ISCO) | → **se descarta** | En los datos es prácticamente **constante** (`'4'` en ~93% y `NaN` donde la ocupación es unknown). Una columna sin variabilidad no discrimina; su ausencia ya está capturada por `es_unknown_ocupacion` |
| `emparejado` | **Nominal** (ok / unknown / rescatado) | **one-hot** (`get_dummies`, 3 columnas) | Son 3 estados sin orden: "rescatado" no es mayor que "ok" |
| `matched_code` | **Nominal** (código con el que se emparejó) | **one-hot en esencia → se fusiona** | Coincide 100% con `occupation_code` cuando `emparejado='ok'` y guarda el código recuperado en `rescatado`; se fusiona en `occupation_code` (ver celda abajo) |
| `occupation_code` | **Nominal** (≈2.856 códigos) | **one-hot** (`get_dummies`) | Códigos de ocupación ISCO son etiquetas sin orden. Ojo: son 2.856 categorías → *explosión de columnas* (se analiza abajo) |
| `occupation_label` | **Nominal** (nombre del cargo) | → **se descarta** | Es 1:1 con `occupation_code` (una etiqueta por código): duplicaría las 2.856 columnas sin información nueva |
| `isco_group` / `isco_group_label` | **Nominal** (grupo ISCO 4 dígitos) | → **se descartan** | `isco_group` es un *agregado* del código de ocupación (sus 4 primeros dígitos) y `isco_group_label` es 1:1 con él; uno-hotear `occupation_code` ya los contiene |
| `es_unknown_ocupacion`, `es_rescatado`, `es_vigente` | **Binarias** (flags True/False) | 0/1 directo (equivale a one-hot con `drop='first'`) | Ya son binarias: escalarlas no aporta y rompe su legibilidad (slide 21) |
| `resume_id` | Identificador | **se descarta** | Es la clave primaria (una por persona), no un patrón |

### 1.1 Verificación de los descartes (auditoría de redundancia)

Los descartes NO son opinión: se demuestran con los datos.


In [3]:
ok = df[df["emparejado"] == "ok"]
print("matched_code == occupation_code cuando emparejado='ok':",
      round(float((ok["matched_code"] == ok["occupation_code"]).mean()), 4))

g = df.dropna(subset=["occupation_code", "occupation_label"])
print("occupation_label x code (max etiquetas por codigo, 1 = 1:1):",
      int(g.groupby("occupation_code")["occupation_label"].nunique().max()))

g2 = df.dropna(subset=["isco_group", "isco_group_label"])
print("isco_group_label x group (max etiquetas por grupo, 1 = 1:1):",
      int(g2.groupby("isco_group")["isco_group_label"].nunique().max()))

print("isco_level valores distintos:", df["isco_level"].dropna().unique().tolist())
print("NaN de isco_level == es_unknown_ocupacion=True:",
      int((df["isco_level"].isna() == (df["es_unknown_ocupacion"] == "True")).all()))

print("\nmatched_code en emparejado='unknown':",
      df.loc[df["emparejado"] == "unknown", "matched_code"].unique()[:3])
print("matched_code en emparejado='rescatado' (codigos recuperados):",
      df.loc[df["emparejado"] == "rescatado", "matched_code"].dropna().unique()[:3])

matched_code == occupation_code cuando emparejado='ok': 1.0


occupation_label x code (max etiquetas por codigo, 1 = 1:1): 1


isco_group_label x group (max etiquetas por grupo, 1 = 1:1): 1
isco_level valores distintos: ['4']
NaN de isco_level == es_unknown_ocupacion=True: 1

matched_code en emparejado='unknown': <ArrowStringArray>
['unknown']
Length: 1, dtype: str
matched_code en emparejado='rescatado' (codigos recuperados): <ArrowStringArray>
['2659.2.2', '3412.4.9', '2424.3']
Length: 3, dtype: str


### 1.2 Fusión: `occupation_code` conserva el código recuperado de `rescatado`

Para `rescatado` el código vive en `matched_code`; para `unknown` ambas están vacías/`'unknown'`. Fusionamos para dejar **una sola** columna de ocupación.


In [4]:
ocup = df["occupation_code"].copy()
fusion = ocup.isna() & (df["matched_code"] != "unknown")
ocup.loc[fusion] = df.loc[fusion, "matched_code"]      # rescatado: trae el codigo recuperado

print("NaN restantes de ocupacion tras fusion:", int(ocup.isna().sum()),
      "| deben coincidir con emparejado='unknown':",
      int((ocup.isna() == (df["emparejado"] == "unknown")).all()))

df = df.assign(occupation_code=ocup)

NaN restantes de ocupacion tras fusion:

 25805 | deben coincidir con emparejado='unknown': 1


---
## 2 · Codificación ordinal

### 2.1 Fechas → ordinal numérico

`Q1 2011` es texto. Lo llevamos a `año × 4 + trimestre` (Q1 2011 = 2011×4+1 = 8.045), que preserva el orden temporal y permite operaciones aritméticas.


In [5]:
RE = r"^Q([1-4])\s+(\d{4})$"

def to_ordinal(x):
    p = x.str.extract(RE)
    return (pd.to_numeric(p[1], errors="coerce") * 4 +
            pd.to_numeric(p[0], errors="coerce"))

start_ord = to_ordinal(df["start_date"])
end_ord   = to_ordinal(df["end_date"])

# 'Present' = censura (el empleo sigue). Lo ubicamos en el trimestre del proyecto (Q1 2026).
presentes = df["end_date"] == "Present"
end_ord = end_ord.mask(presentes, 2026 * 4 + 1)

# Duracion trimestral (feature comportamental derivada de las dos fechas)
dur_q = end_ord - start_ord

print("start_ord : min", start_ord.min(), "max", start_ord.max())
print("end_ord   : min", end_ord.min(), "max", end_ord.max(),
      "| 'Present' reubicados:", int(presentes.sum()))
print("dur_q     : min", dur_q.min(), "max", dur_q.max())

df = df.assign(start_ord=start_ord, end_ord=end_ord, dur_q=dur_q)

start_ord : min 7861 max 8085
end_ord   : min 7881.0 max 8105.0 | 'Present' reubicados: 18956
dur_q     : min 1.0 max 212.0


### 2.2 `university_level` → mapeo ordinal manual

Son **5 niveles discretos con jerarquía real**; un mapeo manual explícito es más fiel que discretizar (KBins agruparía niveles arbitrariamente). Documentamos la escala en el propio código.


In [6]:
ORDEN_UNIV = {
    "No reportado":     0,   # ausencia como estado, no como 0 engañoso de estudios
    "Secondary school": 1,
    "Bachelor":         2,
    "Master":           3,
    "PhD":              4,
}
univ_ord = df["university_level"].map(ORDEN_UNIV)
print("univ_ord unicos:", sorted(univ_ord.unique().tolist()))
print("orden verificado:", [u for u in sorted(univ_ord.unique())])

df = df.assign(univ_ord=univ_ord)

univ_ord unicos: [0, 1, 2, 3, 4]


orden verificado: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


### 2.3 `isco_level`: por qué se descarta (es constante)

Es **ordinal por diseño** (nivel de competencia ISCO 1–4), pero en estos datos no discrimina:
- ~93% de las filas vale `4`; el resto es `NaN` = ocupación desconocida.
- El `NaN` ya lo explica la bandera `es_unknown_ocupacion`.

Dejarlo como número casi constante no aporta información al modelo y genera una falsa sensación de variable.


In [7]:
isco_lv = df["isco_level"]
tabla = isco_lv.fillna("NaN").value_counts()
print("Conteo de isco_level:")
print(tabla.to_string())
prop = tabla.get("4", 0) / len(df)
print(f"\nproporcion de filas == '4': {prop:.4f}  -> columna ~constante, se descarta.")

Conteo de isco_level:
isco_level
4      350762
NaN     25805

proporcion de filas == '4': 0.9315  -> columna ~constante, se descarta.


### 2.4 `KBinsDiscretizer` (método del curso) aplicado a las fechas

El curso lo usa para convertir una continua en **segmentos ordinales** de igual tamaño (*quantile* = ~misma cantidad de observaciones por rango). Lo aplicamos a `start_ord` y `end_ord`.


In [8]:
kbd_inicio = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile")
kbd_fin    = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile")

bins_inicio = kbd_inicio.fit_transform(start_ord.to_frame()).ravel()
bins_fin    = kbd_fin.fit_transform(end_ord.to_frame()).ravel()

print("Bordes de start_ord:", kbd_inicio.bin_edges_[0].round(0))
print("Bordes de end_ord  :", kbd_fin.bin_edges_[0].round(0))
print("\nStart en 5 segmentos (mismo tamano por cuantiles):")
print(pd.Series(bins_inicio).value_counts().sort_index().to_string())

Bordes de start_ord: [7865. 8013. 8037. 8049. 8060. 8082.]
Bordes de end_ord  : [7881. 8029. 8045. 8057. 8067. 8105.]

Start en 5 segmentos (mismo tamano por cuantiles):
0.0    69820
1.0    77191
2.0    72797
3.0    79867
4.0    76892


### 2.5 Las competencias ESCO: `n_skills_essential` y `veces_ese_oficio`

Las 6 columnas de competencias que el pipeline integra llegan como **texto** (el proyecto lee todo
con `dtype=str`, seccion 0). Aqui pasan a numero y se decide cuales entran a la `X`.

**Dos advertencias que condicionan todo lo que sigue** (desarrolladas en `docs/SKILLS.md`):

1. **Atribuyen al CARGO, no a la PERSONA.** ESCO publica que competencias *exige un oficio*, no
   que sabe un individuo: lo que se hace es atribuir al puesto. No hay variabilidad dentro de un
   mismo oficio, asi que esto **no** se puede leer como "esta persona sabe N cosas".
2. **Una parte de las filas no tiene dato.** Cuando el oficio no esta clasificado
   (`saber_skills = 'no_clasificado'`) los contadores vienen **vacios, nunca 0**: un 0 afirmaria
   que el cargo no exige ninguna competencia, que es falso. `sklearn` no acepta `NaN` en un
   escalador y este proyecto no imputa valores, asi que la salida honesta es **dropear esas
   filas** y documentar la perdida. Cuantas son se mide antes de dropear, no despues.

De las 6 columnas entran **dos**. Los otros tres contadores (`_competence`, `_knowledge`,
`_sin_tipo`) son la misma variable repartida y no aportan informacion independiente; y
`n_skills_essential` es funcion determinista de `occupation_code`, o sea que si la `X` ya lleva
`occupation_<codigo>` en one-hot, su informacion ya esta ahi.


In [9]:
# --- 2.5.a Los contadores de ESCO llegan como texto: a numero ---
# Regla del proyecto: todo se lee con dtype=str (para no perder los ceros a la
# izquierda de los codigos ESCO). Los contadores son la excepcion y hay que
# convertirlos explicitamente antes de usarlos como numero.
CONTADORES = ["n_skills_essential", "veces_ese_oficio"]
for c in CONTADORES:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# --- 2.5.b Que se va a perder, medido ANTES de dropear ---
sin_dato = df["n_skills_essential"].isna()
n_antes, p_antes = len(df), df["resume_id"].nunique()

print(f"Filas sin dato de competencias: {int(sin_dato.sum()):,} "
      f"({100 * sin_dato.mean():.2f} %)")
print("  su composicion (emparejado):")
print(df.loc[sin_dato, "emparejado"].value_counts().to_string())
print("\nRegla vazio != 0 -> contadores en 0 (deberia ser 0):",
      int(df["n_skills_essential"].eq(0).sum()))
print("El vacio coincide exacto con saber_skills == 'no_clasificado':",
      bool((sin_dato == (df["saber_skills"] == "no_clasificado")).all()))

# --- 2.5.c El drop, explicito y con su costo medido ---
df = df.loc[~sin_dato].copy()
print(f"\nUniverso de trabajo: {n_antes:,} -> {len(df):,} filas "
      f"({len(df) - n_antes:+,} | {100 * (len(df) - n_antes) / n_antes:+.2f} %)")
print(f"                    {p_antes:,} -> {df['resume_id'].nunique():,} personas")
print(f"Ocupaciones distintas con dato: {df['occupation_code'].nunique():,}")

# --- 2.5.d El binario de reincidencia ---
# El 64 % de los valores de veces_ese_oficio es 1 y su sd es 1,1: como variable
# continua aporta poco mas que "volvio o no". Se conservan las DOS formas porque
# no son lo mismo -> el binario es estable para el escalador, y el conteo
# distingue 2 repeticiones de 19 (que en 32.400 personas es informacion real).
df["reincide"] = (df["veces_ese_oficio"] > 1).astype(np.uint8)
print(f"\nreincide (veces_ese_oficio > 1): {int(df['reincide'].sum()):,} filas "
      f"({100 * df['reincide'].mean():.1f} %) | "
      f"{df.loc[df['reincide'] == 1, 'resume_id'].nunique():,} personas")

print()
for c in CONTADORES + ["reincide"]:
    print(f"  {c:<20} min {df[c].min():>6.0f}  max {df[c].max():>6.0f}  "
          f"mediana {df[c].median():>6.1f}  sd {df[c].std():>6.1f}")


Filas sin dato de competencias: 28,304 (7.52 %)
  su composicion (emparejado):
emparejado
unknown      25805
rescatado     2499

Regla vazio != 0 -> contadores en 0 (deberia ser 0): 0
El vacio coincide exacto con saber_skills == 'no_clasificado': True



Universo de trabajo: 376,567 -> 348,263 filas (-28,304 | -7.52 %)
                    71,061 -> 70,397 personas
Ocupaciones distintas con dato: 2,855

reincide (veces_ese_oficio > 1): 106,250 filas (30.5 %) | 32,400 personas

  n_skills_essential   min      4  max     99  mediana   20.0  sd   13.3
  veces_ese_oficio     min      1  max     19  mediana    1.0  sd    1.1
  reincide             min      0  max      1  mediana    0.0  sd    0.5


---
## 3 · Codificación one-hot con `get_dummies`

### 3.1 El problema de las ~2.856 ocupaciones

> *"ciudad × país explota"* — el curso advierte que pocas columnas nominales de alta cardinalidad generan muchísimas columnas. Una ocupación por persona → filas enormemente dispersas.

El grueso de las filas usa ocupaciones únicas. Por eso:

- La demostración con `get_dummies` (matriz densa) se hace sobre una **submuestra representativa de 50.000 filas**; se mantiene la semilla para reproducibilidad.
- Al final de la sección se muestra la alternativa de **producción**: `OneHotEncoder(sparse_output=True)`, que codifica el dataset completo sin explotar la memoria.

Primero, el **bloque de frecuencias**:


In [10]:
frec = df["occupation_code"].dropna().value_counts()
print("Categorias unicas (con datos):", int(frec.size), "-> ~2.800 columnas dummy\n")
print(frec.head(10).to_string())

idx = RNG.choice(len(df), size=min(SAMPLE_SIZE, len(df)), replace=False)
df_s = df.iloc[idx].reset_index(drop=True)
print(f"\nSubmuestra de trabajo: {len(df_s):,} filas")

Categorias unicas (con datos): 2855 -> ~2.800 columnas dummy

occupation_code
3343.1      16864
5223.4      10017
5223.6       9676
9333.8       7106
5230.1       5990
9412.1       5747
5131.2       5347
9112.2       4785
9329.1       4654
3343.1.7     3675

Submuestra de trabajo: 50,000 filas


In [11]:
# one-hot nominal
dummies_emparejado = pd.get_dummies(df_s["emparejado"], prefix="emparejado", dtype=np.uint8)

# one-hot ocupacion (NaN = sin_ocupacion queda en todo 0 y es explicada por la bandera)
dummies_ocupacion  = pd.get_dummies(df_s["occupation_code"], prefix="occupation", dtype=np.uint8)

# flags binarias -> 0/1 (ya son one-hot de k-1, no requieren get_dummies ni escalado)
flags = (df_s[["es_unknown_ocupacion", "es_rescatado", "es_vigente"]]
         .replace({"True": 1, "False": 0})
         .astype(np.uint8))

print("emparejado ->", dummies_emparejado.shape[1], "columnas",
      dummies_emparejado.columns.tolist())
print("ocupacion  ->", dummies_ocupacion.shape[1], "columnas")
print("flags      ->", flags.shape[1], "columnas", flags.columns.tolist())

emparejado -> 1 columnas ['emparejado_ok']
ocupacion  -> 2316 columnas
flags      -> 3 columnas ['es_unknown_ocupacion', 'es_rescatado', 'es_vigente']


> **Recuerda (slides 18–21 de la sesión 4):** las columnas dummy y binarias **NO se escalan**. Escalarlas las vuelve ilegibles sin ganar nada.

### 3.2 Alternativa de producción: `OneHotEncoder` con salida dispersa (dataset completo)

Mismo resultado, pero usa una matriz *sparse* (guarda solo los 1). Con esto **sí** se puede codificar el dataset completo sin 1–9 GB en RAM.


In [12]:
ohe = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X_ohe_full = ohe.fit_transform(df[["emparejado", "occupation_code"]])
print("OneHotEncoder (full, sparse):", X_ohe_full.shape)
print("Memoria relativa del bloque cf. densa:", X_ohe_full.data.nbytes, "bytes aprx.")

OneHotEncoder (full, sparse): (348263, 2856)
Memoria relativa del bloque cf. densa: 5572208 bytes aprx.


---
## 4 · Escalamiento: ¿cuál de los 3 y por qué?

### 4.1 Los outliers documentados en la sesión 3 (auditoría en tiempo de ejecución)

La sesión 3 trató cada versión de forma distinta: en la versión **con** cola larga conservó los extremos (decisión de negocio); en la versión **sin** cola larga los eliminó (`dur ≥ 25` trimestres). Aquí se re-calculan los límites Tukey (IQR) **sobre la versión elegida**, en las dos variantes de `dur_q` (con `Present` → censura, igual que la sesión 3, y con `Present` → Q1 2026, nuestro encoding):


In [13]:
def tukey(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    li, ls = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return q1, q3, iqr, li, ls, int(((s < li) | (s > ls)).sum())

dur_q_ref = (to_ordinal(df["end_date"]) - start_ord)   # Present -> NaN como sesion 3

filas = []
for nombre, var in [("start_ord", start_ord), ("end_ord", end_ord),
                    ("dur_q (Present censurado)", dur_q_ref),
                    ("dur_q (Present = Q1 2026)", dur_q),
                    ("university_map", univ_ord)]:
    q1, q3, iqr, li, ls, out = tukey(var.dropna())
    filas.append([nombre, round(q1, 1), round(q3, 1), round(iqr, 1),
                  round(li, 1), round(ls, 1), out, round(out / len(df), 3)])

pd.DataFrame(filas, columns=["variable", "Q1", "Q3", "IQR", "lim_inf",
                             "lim_sup", "outliers", "prop_outliers"])

,variable,Q1,Q3,IQR,lim_inf,lim_sup,outliers,prop_outliers
0,start_ord,8021.0,8057.0,36.0,7967.0,8111.0,20962,0.060
1,end_ord,8033.0,8065.0,32.0,7985.0,8113.0,17993,0.052
2,dur_q (Present censurado),1.0,11.0,10.0,-14.0,26.0,26505,0.076
3,dur_q (Present = Q1 2026),1.0,12.0,11.0,-15.5,28.5,41410,0.119
4,university_map,1.0,2.0,1.0,-0.5,3.5,1652,0.005


### 4.2 Decisiones de escalador (una por columna)

**Por qué escalar (medido, no supuesto).** KNN y K-means eligen vecinos por distancia euclídea, y la distancia suma diferencias **al cuadrado**: la columna con más unidades secuestra la métrica y las demás quedan mudas. La celda siguiente lo cuantifica sobre nuestra submuestra: cuánto aporta cada columna a la distancia entre `N_PARES` parejas de filas, sin escalar, con cada escalador por separado y con la mezcla que sale a producción. No lo afirmamos de teoría, lo calculamos.

**Lo que revela la Medición.** Escalar no es solo «poner en la misma unidad»: es **decidir a qué variable el modelo le va a hacer caso**. Sin escala, las dos fechas se llevan 80,9 % de la distancia y la educación queda en 0,03 % — un agrupamiento separaría por calendario, no por carrera. Con la mezcla de producción, la duración del empleo (nuestro comportamiento de interés) queda en 32,51 %, y las competencias ESCO —que son justo lo que se agregó en la 2.5— pasan de 8,19 % sin escalar a 17,68 % con su `RobustScaler`.

**Deuda que deja la auditoría (no se corrige en silencio).** `MinMaxScaler` lleva la educación (rango 0–4) a [0, 1], pero `RobustScaler` extiende las fechas por IQR, que es mucho más angosto que su rango total; el resultado es que en la distancia la educación queda por debajo del 1 %. Elegimos cada escalador por el perfil de outliers de su columna, no por su aporte a la distancia: es un criterio válido, pero tiene este efecto lateral. Se documenta como decisión abierta a revisar antes de la fase de minería, junto con D1–D4.

| Columna | Outliers detectados | Escalador | Por qué |
|---|---|---|---|
| `start_ord` | ~21 mil (cola baja: empleos iniciados en los 60s–70s) | **RobustScaler** | La mediana y el IQR no se dejan arrastrar por la cola baja; conserva la estructura del calendario |
| `end_ord` | ~18 mil (cola baja similar, más la censura de `Present`) | **RobustScaler** | Idem |
| `dur_q` | cola larga real (~41 mil en nuestro encoding; 35.477 según la sesión 3 con censura) | **RobustScaler** | Es el fenómeno documentado en la sesión 3: empleos de décadas. Robust lo "aísla" |
| `university_map` | rango fijo [0, 4] por diseño, sin atípicos | **MinMaxScaler** | Rango acotado y conocido → el caso natural del MinMax; lleva todo a [0, 1] |
| `n_skills_essential` | sin atípicos (rango 4–99) | **RobustScaler** | Decisión por **escala**, no por outliers: con 94 unidades de rango, sin escalarla duplica el peso de las dos fechas y deja muda a `univ_ord` (rango 4). Se mide en la celda siguiente |
| `veces_ese_oficio` | sin atípicos (rango 1–19) | **RobustScaler** | Idem. Además está muy concentrada (64 % de valores = 1), de ahí que se le acompaña su binario |
| `reincide` (`veces > 1`) | — | *no se escala* | Es 0/1: ya está en su rango. Escalarla la haría ilegible sin ganar nada (misma regla que las dummies, slide 21) |
| `StandardScaler` | — | *no se elige* | Usa media/desviación: con la cola larga/las fechas viejas la media se corrompe (sensibilidad moderada). Se muestra la comparación abajo |

> **Fuga de datos (slide 21 · Error 1):** el escalador se ajusta (`fit`) **solo** sobre la muestra de trabajo; jamás sobre los datos de prueba. Aquí lo hacemos explícito.


In [14]:
# 1) Robust para fechas y duracion
rob = RobustScaler()
escaladas_rob = pd.DataFrame(
    rob.fit_transform(df_s[["start_ord", "end_ord", "dur_q"]].to_numpy()),
    columns=["start_ord_rob", "end_ord_rob", "dur_q_rob"], index=df_s.index)

# 2) MinMax para education (rango acotado sin atipicos)
mm = MinMaxScaler()
univ_mm = pd.DataFrame(mm.fit_transform(df_s[["univ_ord"]].to_numpy()),
                       columns=["university_mm"], index=df_s.index)

print("Robust -> mediana despues de escalar (debe ser ~0):",
      escaladas_rob.median().round(4).tolist())
print("MinMax -> min/max despues de escalar (deben ser 0/1):",
      (univ_mm.min().round(4).values[0], univ_mm.max().round(4).values[0]))

# 2b) Robust para las competencias ESCO. Ojo: esta decision NO es por outliers
# como las fechas, es por ESCALA. n_skills_essential tiene rango 4-99 y
# veces_ese_oficio 1-19: sin escalarlas, su peso en la distancia aplasta a
# univ_ord (rango 0-4). La celda siguiente lo cuantifica.
# El binario `reincide` NO se escala: ya es 0/1 (regla de dummies y binarias).
rob_sk = RobustScaler()
skills_rob = pd.DataFrame(
    rob_sk.fit_transform(df_s[CONTADORES].to_numpy()),
    columns=[c + "_rob" for c in CONTADORES], index=df_s.index)

print("Robust (competencias) -> mediana despues de escalar (debe ser ~0):",
      skills_rob.median().round(4).tolist())
print("Sin escalar `reincide` porque ya es 0/1; valores:",
      sorted(df_s["reincide"].unique().tolist()))

# 3) Comparacion de los tres escaladores sobre dur_q (10 observaciones)
durq_demo = df_s["dur_q"].to_numpy().reshape(-1, 1).astype(float)
tabla = pd.DataFrame({
    "crudo":      durq_demo[:, 0],
    "minmax":     MinMaxScaler().fit_transform(durq_demo)[:, 0],
    "standard":   StandardScaler().fit_transform(durq_demo)[:, 0],
    "robust":     RobustScaler().fit_transform(durq_demo)[:, 0],
})
print("\nEscaladores sobre dur_q (10 primeras de la muestra):")
print(tabla.head(10).round(3).to_string())
print("\nEl robust mantiene la estructura de la distribucion; el minmax y el")
print("standard quedan a merced de los extremos (cola).")

Robust -> mediana despues de escalar (debe ser ~0): [0.0, 0.0, 0.0]
MinMax -> min/max despues de escalar (deben ser 0/1): (np.float64(0.0), np.float64(1.0))
Robust (competencias) -> mediana despues de escalar (debe ser ~0): [0.0, 0.0]
Sin escalar `reincide` porque ya es 0/1; valores: [0, 1]

Escaladores sobre dur_q (10 primeras de la muestra):
   crudo  minmax  standard  robust
0    4.0   0.014    -0.411   0.000
1   16.0   0.071     0.227   1.091
2   76.0   0.355     3.416   6.545
3    1.0   0.000    -0.571  -0.273
4    4.0   0.014    -0.411   0.000
5   24.0   0.109     0.652   1.818
6    4.0   0.014    -0.411   0.000
7    1.0   0.000    -0.571  -0.273
8   24.0   0.109     0.652   1.818
9   16.0   0.071     0.227   1.091

El robust mantiene la estructura de la distribucion; el minmax y el
standard quedan a merced de los extremos (cola).


In [15]:
# 4) POR QUE ESCALAR: cuanto aporta cada columna a la distancia euclidea.
# Espejo del bloque 1 de transformaciones.py (material del docente, sesion 4).
# KNN y K-means eligen vecinos por distancia; como la distancia suma
# diferencias al cuadrado, la columna con mas unidades secuestra la metrica.
# Columnas que entran a la medicion de aporte a la distancia.
# Se separan por escalador, y por eso se listan por grupo y no en un solo bloque.
NUM     = ["start_ord", "end_ord", "dur_q"]          # Robust (colas reales)
SKILLS  = ["n_skills_essential", "veces_ese_oficio"]  # Robust (escala incompatible, ver 2.5)
BINARIA = ["reincide"]                               # 0/1 -> sin escalar


def aporte_porcentual(Z, pares):
    """% que aporta cada columna a la distancia euclidea, promedio de los pares.

    Como la distancia suma diferencias al cuadrado, el aporte de cada columna
    es proporcional a la suma de sus diferencias al cuadrado.
    """
    dif = (Z[pares[:, 0]] - Z[pares[:, 1]]) ** 2
    return 100 * dif.sum(axis=0) / dif.sum()


rng_dist = np.random.RandomState(SEED_DIST)
pares = rng_dist.randint(0, len(df_s), size=(N_PARES, 2))

TODAS = NUM + SKILLS + ["univ_ord"] + BINARIA
escenarios = {
    "CRUCO (sin escalar)": df_s[TODAS].to_numpy(float),
    "StandardScaler": StandardScaler().fit_transform(df_s[TODAS]),
    "MinMaxScaler": MinMaxScaler().fit_transform(df_s[TODAS]),
    "RobustScaler": RobustScaler().fit_transform(df_s[TODAS]),
    # La mezcla que realmente sale a produccion (seccion 5): los dos escaladores
    # ya ajustados arriba, cada uno sobre las columnas que le corresponden.
    # Se pasan como ndarray porque en la celda anterior se ajustaron sobre .to_numpy().
    "PRODUCCION (rob+mm)": np.hstack([rob.transform(df_s[NUM].to_numpy()),
                                      rob_sk.transform(df_s[SKILLS].to_numpy()),
                                      mm.transform(df_s[["univ_ord"]].to_numpy()),
                                      df_s[BINARIA].to_numpy(float)]),
}

tabla_aporte = pd.DataFrame(
    {nombre: aporte_porcentual(Z, pares) for nombre, Z in escenarios.items()},
    index=TODAS,
).T.round(2)
tabla_aporte.columns = TODAS

print("Aporte de cada columna a la distancia euclidea (%)")
print(f"Sobre {N_PARES} parejas aleatorias de la submuestra de {len(df_s):,} filas\n")
print(tabla_aporte.to_string())

fechas_crudo = tabla_aporte.loc["CRUCO (sin escalar)", ["start_ord", "end_ord"]].sum()
print(f"\nSin escalar, las dos fechas aportan {fechas_crudo:.1f} % de la distancia")
print(f"y la educacion solo {tabla_aporte.loc['CRUCO (sin escalar)', 'univ_ord']:.2f} %:")
print("  -> sin escala, KNN/K-means agruparían por CALENDARIO, no por carrera.")
print(f"\nCon la mezcla de produccion, la duracion del empleo pasa de "
      f"{tabla_aporte.loc['CRUCO (sin escalar)', 'dur_q']:.1f} % a "
      f"{tabla_aporte.loc['PRODUCCION (rob+mm)', 'dur_q']:.1f} % de la distancia.")
print(f"Pero la educacion queda en {tabla_aporte.loc['PRODUCCION (rob+mm)', 'univ_ord']:.2f} %:")
print("  -> efecto lateral de escalar las fechas por IQR (Robust) y la educacion por")
print("     rango total (MinMax). Se documenta como deuda abierta, no se cambia en")
print("     silencio: revisar antes de la fase de mineria (ver seccion 5).")

# El caso concreto que motivo agregar las competencias: sin escalarlas, una
# columna con 94 unidades de rango deja muda a otra que tiene 4.
print("")
print("Aporte de las competencias ESCO a la distancia (seccion 2.5):")
for col in SKILLS:
    print(f"  {col:<20} sin escalar {tabla_aporte.loc['CRUCO (sin escalar)', col]:>6.2f} %"
          f"   ->   con Robust {tabla_aporte.loc['PRODUCCION (rob+mm)', col]:>6.2f} %")
print(f"  {'reincide (0/1)':<20} sin escalar "
      f"{tabla_aporte.loc['CRUCO (sin escalar)', 'reincide']:>6.2f} %"
      f"   ->   en produccion {tabla_aporte.loc['PRODUCCION (rob+mm)', 'reincide']:>6.2f} %")
print("  -> el binario no necesita escalador: ya esta en [0, 1]. Aplicarselo lo")
print("     achataria sin ganar nada (regla de dummies y binarias, slide 21).")

# Una pareja concreta: mismo nivel educativo, duraciones lo mas distintas posible.
nivel, dur = df_s["univ_ord"].to_numpy(), df_s["dur_q"].to_numpy()
mejor = None
for n in np.unique(nivel):
    pos = np.flatnonzero(nivel == n)
    if len(pos) < 2:
        continue
    d = dur[pos]
    cand = (d.max() - d.min(), pos[np.argmax(d)], pos[np.argmin(d)])
    if mejor is None or cand[0] > mejor[0]:
        mejor = cand

ia, ib = mejor[1], mejor[2]
print(f"\nPareja concreta (mismo nivel educativo, duraciones opuestas):")
print(f"  A: {df_s['start_date'].iloc[ia]} -> {df_s['end_date'].iloc[ia]} | dur_q = {dur[ia]:.0f} | "
      f"{df_s['university_level'].iloc[ia]} | {df_s['occupation_label'].iloc[ia]}")
print(f"  B: {df_s['start_date'].iloc[ib]} -> {df_s['end_date'].iloc[ib]} | dur_q = {dur[ib]:.0f} | "
      f"{df_s['university_level'].iloc[ib]} | {df_s['occupation_label'].iloc[ib]}")

print("\nDistancia euclidea entre A y B:")
for nombre, Z in escenarios.items():
    print(f"  {nombre:<22}{np.sqrt(((Z[ia] - Z[ib]) ** 2).sum()):>10.3f}")

Aporte de cada columna a la distancia euclidea (%)
Sobre 300 parejas aleatorias de la submuestra de 50,000 filas

                     start_ord  end_ord  dur_q  n_skills_essential  veces_ese_oficio  univ_ord  reincide
CRUCO (sin escalar)      42.94    38.00  10.78                8.19              0.05      0.03      0.01
StandardScaler           14.38    14.10  10.70               16.28             14.61     12.83     17.09
MinMaxScaler              5.66     5.19   1.60                6.13              1.10     12.05     68.26
RobustScaler             11.00    12.32  29.56               16.08             17.95      9.67      3.42
PRODUCCION (rob+mm)      12.09    13.54  32.51               17.68             19.74      0.66      3.77

Sin escalar, las dos fechas aportan 80.9 % de la distancia
y la educacion solo 0.03 %:
  -> sin escala, KNN/K-means agruparían por CALENDARIO, no por carrera.

Con la mezcla de produccion, la duracion del empleo pasa de 10.8 % a 32.5 % de la distancia.
Pe

### 4.3 ¿PCA? Auditoría de correlación (slides 12–14)

El pipeline de la sesión 4 cierra con **"PCA, si aplica"**: la clase pide una **correlación** antes de decidir (`pca si o no aplica hay que hacer una correlacion`). PCA busca pocas direcciones (componentes) que expliquen la mayor varianza de columnas **numéricas escaladas**; si las columnas están poco correlacionadas, no hay direcciones que comprimir y PCA **no aplica**.

Aquí lo auditamos con honestidad:

1. **Escalamos** las numéricas con `StandardScaler` (el requisito del slide 12 es PCA sobre datos escalados; caso donde Standard **sí** corresponde).
2. **Correlaciones** de Pearson entre las numéricas.
3. **Varianza acumulada** de las componentes: necesitamos ~80–90 % para justificar comprimir.
4. Las **dummies** (one-hot) quedan fuera: son categóricas puras y ya se "comprimió" convirtiendo categorías en columnas; PCA sobre 2.800 dummies araña información estructural (slide 14).
5. Las **competencias ESCO sí entran** (2 columnas de la 2.5) porque son numéricas continuas. El binario `reincide` queda fuera, por la misma lógica que las dummies: es un indicador 0/1, no una dimensión.


In [16]:
# 1) Matriz de correlacion de las numericas de la muestra.
# Las competencias entran (seccion 2.5); el binario `reincide` NO, por la misma
# razon que las dummies: es una indicador categorico, no una dimension continua.
NUM_PCA = ["start_ord", "end_ord", "dur_q", "univ_ord"] + SKILLS
X_num = df_s[NUM_PCA].astype(float)
print("Numericas que entran a PCA:", X_num.shape[1], "columnas")
print("Correlacion de Pearson (numericas de la muestra):")
print(X_num.corr().round(3).to_string())
print()
print("Nota: dur_q = end_ord - start_ord  ->  tres de las columnas guardan una")
print("relacion lineal casi exacta. Las competencias, en cambio, son ortogonales")
print("a las fechas: describen el PUESTO, no el CUANDO.")

# 2) PCA sobre las numericas estandarizadas (StandardScaler = prerequisito del slide 12)
Xp_std = StandardScaler().fit_transform(X_num)
pca = PCA(random_state=RNG)
Xp = pca.fit_transform(Xp_std)
var_ac = np.cumsum(pca.explained_variance_ratio_)
k = X_num.shape[1]
df_var = pd.DataFrame({
    "componente": range(1, k + 1),
    "varianza_individual": np.round(pca.explained_variance_ratio_[:k], 4),
    "varianza_acumulada": np.round(var_ac[:k], 4)},
).set_index("componente")
print(f"\nVarianza explicada del PCA sobre las {k} numericas:")
print(df_var.to_string())
print(f"\nPara llegar a ~80% se necesitan {int((var_ac < 0.80).sum()) + 1} de {k} componentes")

# 3) ?Que pasa con las dummies y el binario? Solo contamos columnas: no entran a PCA
print("\nColumnas one-hot de occupation_code:", dummies_ocupacion.shape[1], "-> no entran a PCA")
print("Binario reincide: 1 columna -> tampoco entra (misma logica que las dummies)")
print("Diagnostico: PCA casi no comprime aqui. No se aplica a X final.")


Numericas que entran a PCA: 6 columnas
Correlacion de Pearson (numericas de la muestra):
                    start_ord  end_ord  dur_q  univ_ord  n_skills_essential  veces_ese_oficio
start_ord               1.000    0.824 -0.374     0.114               0.046             0.040
end_ord                 0.824    1.000  0.217     0.091               0.036            -0.004
dur_q                  -0.374    0.217  1.000    -0.047              -0.020            -0.076
univ_ord                0.114    0.091 -0.047     1.000               0.106            -0.031
n_skills_essential      0.046    0.036 -0.020     0.106               1.000             0.098
veces_ese_oficio        0.040   -0.004 -0.076    -0.031               0.098             1.000

Nota: dur_q = end_ord - start_ord  ->  tres de las columnas guardan una
relacion lineal casi exacta. Las competencias, en cambio, son ortogonales
a las fechas: describen el PUESTO, no el CUANDO.

Varianza explicada del PCA sobre las 6 numericas:
      

**Conclusión de la auditoría (misma lógica del resultado de TelecomUNO en el slide 13):** PCA **no comprime** en este dataset. Con 6 numéricas hacen falta **4 de 6 componentes** para llegar al 80 % de la varianza, y la última aporta 0,0000 de varianza individual: el espectro está repartido, no concentrado.

El motivo es que las columnas miden cosas distintas y no se solapan: las fechas y la duración son una **relación lineal casi exacta** entre sí (`dur_q` = `end_ord` − `start_ord`), y las competencias ESCO son **ortogonales** a todo eso — describen el *puesto*, no el *cuándo*. `univ_ord` también aporta su propia dimensión. No hay una dirección dominante que capture la estructura, que es justamente la condición para que PCA sirva.

`PCA` se **descarta** entonces, y por diseño queda excluido de la matriz final (dummies inmensas + columna densa pequeña). Su valor aquí fue **diagnóstico**: confirmó que no hay una estructura de varianza compartida que comprimir sin romper el one-hot.


---
## 5 · Matriz X final lista para modelar

**Formato (espejo de la sesión 4, slide 16):** numéricas/ordinales escaladas + dummies + banderas. Nada de texto, nada de binarias escaladas, sin `resume_id`, sin columnas descartadas.


In [17]:
X_final = pd.concat([
    escaladas_rob,                     # start_ord_rob, end_ord_rob, dur_q_rob
    skills_rob,                        # n_skills_essential_rob, veces_ese_oficio_rob
    univ_mm,                           # university_mm
    df_s[["reincide"]],                # 0/1, sin escalar
    dummies_emparejado,                # emparejado_ok / _unknown / _rescatado
    dummies_ocupacion,                 # occupation_<codigo>
    flags,                             # es_unknown_ocupacion, es_rescatado, es_vigente
], axis=1)

print("X final (submuestra):", X_final.shape)
print("\nPrimeras columnas:", X_final.columns[:12].tolist(), "...")
print("\nComprobaciones de sanidad:")
print(" - dtypes no-numericos:", int(X_final.select_dtypes(exclude=["number"]).shape[1]))
print(" - nulos:", int(X_final.isna().sum().sum()))
X_final.head(3)

X final (submuestra): (50000, 2327)

Primeras columnas: ['start_ord_rob', 'end_ord_rob', 'dur_q_rob', 'n_skills_essential_rob', 'veces_ese_oficio_rob', 'university_mm', 'reincide', 'emparejado_ok', 'occupation_0110.10', 'occupation_0110.11', 'occupation_0110.12', 'occupation_0110.2'] ...

Comprobaciones de sanidad:
 - dtypes no-numericos: 0


 - nulos: 0


,start_ord_rob,end_ord_rob,dur_q_rob,n_skills_essential_rob,veces_ese_oficio_rob,university_mm,reincide,emparejado_ok,occupation_0110.10,occupation_0110.11,...,occupation_9629.1,occupation_9629.2,occupation_9629.3,occupation_9629.4,occupation_9629.5,occupation_9629.6,occupation_9629.7,es_unknown_ocupacion,es_rescatado,es_vigente
0,0.388889,0.3125,0.000000,-1.000000,0.0,0.50,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,-0.388889,-0.1875,1.090909,0.000000,5.0,0.25,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,-1.833333,0.0625,6.545455,0.307692,1.0,0.50,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0


> La variable objetivo `y` aún no existe: la minería de **transiciones laborales** (qué ocupaciones siguen a cuáles por persona) se formaliza en la siguiente fase. Por ahora entregamos la **X** que habilitará KNN / K-means / árboles en sesiones 7–9.

### Guardar (opcional)

Descomente la siguiente celda si quiere persistir la matriz. Nota: 50.000×~2.800 columnas densas ocupan ~100–150 MB; para el dataset completo use el bloque `OneHotEncoder` sparse.


In [18]:
# X_final.to_csv(PROC / f"X_empleos_{VERSION}_transformado_sample.csv", index=False)
# print("Guardado:", PROC / f"X_empleos_{VERSION}_transformado_sample.csv")

---
## 6 · Resumen del pipeline y cierre

```text
empleos_limpio.csv (376.567 (con cola larga) x 20)
  ├─ 1  Ordinal:  fechas        -> anio*4+q -> KBinsDiscretizer (demo) + RobustScaler
  ├─ 2  Ordinal:  university_level -> mapeo manual 0-4 -> MinMaxScaler
  ├─ 3  One-hot:  emparejado (3) + occupation_code (~2.856)  [get_dummies demo / OHE-sparse prod]
  ├─ 4  Binarias: es_unknown_ocupacion, es_rescatado, es_vigente (0/1, sin escalar)
  ├─ 5  Derivada: dur_q (duracion trimestral) -> RobustScaler
  ├─ 6  ESCO 2.5: n_skills_essential, veces_ese_oficio -> RobustScaler
  │        reincide (veces > 1) -> 0/1 sin escalar
  │        [se descartan 28.304 filas sin oficio clasificado: 376.567 -> 348.263]
  └─ -  Descartadas (justificado): resume_id, matched_code (fusionado), occupation_label
      (1:1), isco_group / isco_group_label (agregados), isco_level (constante)
```

**Decisiones clave de esta sesión**

1. `isco_level` es ordinal por diseño pero **constante** en los datos → descartada (su `NaN` = `es_unknown_ocupacion`).
2. Redundancias demostradas → `matched_code`, `occupation_label`, `isco_group*` descartados tras auditoría (1:1 / prefijo / fusión).
3. Escaladores elegidos **por los outliers re-auditados en la sección 4.1**: robust para fechas y duración (colas reales), **minmax** para educación (rango acotado sin atípicos) y standard **descartado** por sensibilidad a las colas.
4. Dummies y banderas binarias **no se escalan** (slide 21). `reincide` tampoco: ya es 0/1.
5. **Competencias ESCO (2.5):** de las 6 columnas entran 2 a la `X` más un binario. `n_skills_essential` es función determinista de `occupation_code` (0 de 2.855 oficios con más de un valor), así que su información ya está en el one-hot de ocupación; y `n_skills_competence` / `_knowledge` / `_sin_tipo` son la misma variable repartida, no tres. Se **descartan** de la `X` (siguen en el CSV).
6. **Las 28.304 filas sin oficio clasificado (7,52 %) salen de la `X`**: sus contadores están vacíos —nunca en 0, porque 0 afirmaría que el cargo no exige competencia— y `sklearn` no admite `NaN` en un escalador. El proyecto no imputa, así que se dropean y se declara la pérdida: 376.567 → 348.263 filas, 71.061 → 70.397 personas.
7. `fit` del escalador solo sobre la muestra de trabajo (evita fuga de datos).
8. **PCA auditado y descartado (sección 4.3):** hacen falta 4 de 6 componentes para el 80 % de la varianza y la última no aporta nada, así que no hay una dirección dominante que comprimir. Las fechas y la duración son una dependencia lineal entre sí, y las competencias ESCO son ortogonales a ellas (`n_skills_essential` vs `start_ord` = 0,046). Las dummies y el binario no entran por diseño. No se aplica a `X` final.

**Comparación con la otra versión (4.0 / 4.1):** el pipeline es idéntico; lo que cambia es la magnitud de los outliers y, por tanto, la justificación del robust en `dur_q`.

**Siguiente paso:** sesión 5 (EDA: histogramas, correlaciones, mapa de calor) sobre esta X y las banderas.
